# Lecture: Generative Adversarial Networks (GANs)

In the previous notebook we saw that the VAE produces **blurry** samples. The reason is the MSE reconstruction loss: it penalises the average pixel-wise distance to the training image, which forces the decoder to produce the *mean* over all plausible outputs — a blurry compromise.

A **Generative Adversarial Network** (Goodfellow et al., 2014) takes a fundamentally different approach. Instead of a fixed loss function, it introduces a second network — the **Discriminator** $D$ — that acts as a learned critic:

- **Generator** $G$: maps a noise vector $z \sim \mathcal{N}(0, I)$ to a fake image $G(z)$.
- **Discriminator** $D$: maps an image $x$ to the probability $D(x) \in [0, 1]$ that it is real.

The two networks play a **minimax game**:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))]$$

- $D$ tries to **maximise** the objective: output high values for real images, low values for fakes.
- $G$ tries to **minimise** the objective: produce images that $D$ cannot distinguish from real ones.

In practice the generator minimises $-\mathbb{E}[\log D(G(z))]$ (non-saturating loss) instead of $\mathbb{E}[\log(1 - D(G(z)))]$, which provides stronger gradients early in training when $D$ easily rejects all fakes.

The key insight: $D$ never sees a pixel-wise distance — it only judges *realism*. This pushes $G$ to produce **sharp, realistic-looking images** rather than blurry averages.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/GAN.py ./

### Data Preparation

We train on **Fashion-MNIST** — the same dataset used at the end of the VAE notebook. This allows a direct visual comparison of sample quality between VAE and GAN.

Images are normalised to $[-1, 1]$ to match the generator's Tanh output activation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from GAN import GAN

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_DIM = 64
CHANNELS   = 64
BATCH_SIZE = 256
EPOCHS     = 50
LR         = 2e-4

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

# Normalise to [-1, 1] to match the generator's Tanh output
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")

### Model Architecture

**Generator** $G$: Takes a noise vector $z \in \mathbb{R}^{64}$ and produces a 28×28 image via a linear projection followed by three transposed convolutional blocks. Tanh constrains output to $[-1, 1]$.

**Discriminator** $D$: Takes a 28×28 image and outputs a single probability via three strided convolutional blocks and a linear head. LeakyReLU (slope 0.2) is used instead of ReLU — a standard choice for discriminators that prevents dead neurons without the mode-collapse risk associated with ReLU.

Note that the discriminator has **no BatchNorm in the first layer**: adding normalisation before the first conv would destroy the scale information that helps $D$ detect generated images early in training.

In [ ]:
model = GAN(latent_dim=LATENT_DIM, channels=CHANNELS).to(device)

# Verify output shapes
_z    = torch.randn(4, LATENT_DIM)
_fake = model.generator(_z)
_prob = model.discriminator(_fake)

print("Generator output shape:     ", _fake.shape)  # (4, 1, 28, 28)
print("Discriminator output shape: ", _prob.shape)  # (4, 1)

n_params_g = sum(p.numel() for p in model.generator.parameters())
n_params_d = sum(p.numel() for p in model.discriminator.parameters())
print(f"Generator parameters:       {n_params_g:,}")
print(f"Discriminator parameters:   {n_params_d:,}")

### Training

GAN training alternates between two update steps per mini-batch:

**Step 1 — Update Discriminator:**
$$\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1 - D(G(z)))]$$
Real images should be classified as real ($D(x) \to 1$), generated images as fake ($D(G(z)) \to 0$).

**Step 2 — Update Generator:**
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$$
The generator is updated with the discriminator weights **frozen** (`detach()` on the fake images prevents gradients from flowing back into $D$). $G$ learns to produce images that $D$ classifies as real.

Both losses are logged separately each epoch. In a well-training GAN, $\mathcal{L}_D \approx \log 2 \approx 0.69$ at equilibrium — $D$ cannot do better than random guessing.

In [ ]:
criterion = nn.BCELoss()

opt_d = optim.Adam(model.discriminator.parameters(), lr=LR, betas=(0.5, 0.999))
opt_g = optim.Adam(model.generator.parameters(),     lr=LR, betas=(0.5, 0.999))

# Fixed noise for tracking generator progress across epochs
fixed_z = torch.randn(16, LATENT_DIM, device=device)

history_d, history_g = [], []

for epoch in range(EPOCHS):
    model.train()
    total_d = total_g = 0

    for real, _ in train_loader:
        real = real.to(device, non_blocking=True)
        B    = real.size(0)

        real_labels = torch.ones(B,  1, device=device)
        fake_labels = torch.zeros(B, 1, device=device)

        # ---- Step 1: Update Discriminator --------------------------------
        z    = torch.randn(B, LATENT_DIM, device=device)
        fake = model.generator(z).detach()  # detach: no gradient into G

        loss_d_real = criterion(model.discriminator(real), real_labels)
        loss_d_fake = criterion(model.discriminator(fake), fake_labels)
        loss_d      = (loss_d_real + loss_d_fake) / 2

        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        # ---- Step 2: Update Generator ------------------------------------
        z    = torch.randn(B, LATENT_DIM, device=device)
        fake = model.generator(z)

        # Generator wants D to output 1 (real) for its fakes
        loss_g = criterion(model.discriminator(fake), real_labels)

        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        total_d += loss_d.item()
        total_g += loss_g.item()

    n = len(train_loader)
    history_d.append(total_d / n)
    history_g.append(total_g / n)
    print(f"Epoch {epoch+1:3d}  loss_D={history_d[-1]:.4f}  loss_G={history_g[-1]:.4f}")

In [ ]:
model.save_model()

If you do not want to train, load the pre-trained model (latent_dim=64, 50 epochs, full Fashion-MNIST training set).

In [ ]:
model = GAN(latent_dim=LATENT_DIM, channels=CHANNELS).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/gan_fashion_mnist.pth", device=device)

### Training Dynamics

Unlike VAE training, GAN losses do **not** simply decrease over time. The discriminator and generator are constantly adapting to each other, which produces characteristic oscillating loss curves.

At equilibrium, theory predicts $\mathcal{L}_D \approx \log 2 \approx 0.693$ — the point where the discriminator cannot distinguish real from fake better than random chance. In practice this equilibrium is rarely perfectly stable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_d, label="Discriminator loss $\\mathcal{L}_D$")
ax.plot(history_g, label="Generator loss $\\mathcal{L}_G$")
ax.axhline(np.log(2), color="gray", linestyle="--", label="Equilibrium ($\\log 2 \\approx 0.693$)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("GAN Training Dynamics — Fashion-MNIST")
ax.legend()
plt.tight_layout()
plt.show()

### Generated Samples

We sample 16 noise vectors from $p(z) = \mathcal{N}(0, I)$ and decode them with the trained generator. Images are rescaled from $[-1, 1]$ to $[0, 1]$ for display.

Compare these to the VAE samples from the previous notebook: GAN samples are **sharper** and show crisper edges and textures. The trade-off is that GANs offer no latent space with the structured properties of the VAE posterior.

In [ ]:
model.eval()

samples = model.generate(16, device=device).cpu()
samples = (samples + 1) / 2  # [-1, 1] -> [0, 1]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("GAN samples from $p(z) = \\mathcal{N}(0, I)$ — Fashion-MNIST", y=1.02)
plt.tight_layout()
plt.show()

### VAE vs. GAN — Direct Comparison

Side-by-side comparison of samples from the Fashion-MNIST VAE (notebook 65) and the GAN trained above.

| | VAE | GAN |
|---|---|---|
| Sample quality | Blurry (MSE averages over pixels) | Sharp (adversarial loss rewards realism) |
| Latent space | Structured, continuous, interpolatable | Unstructured — no posterior to inspect |
| Training | Stable, single objective | Unstable, two competing objectives |
| Mode coverage | Good (KL regularisation) | Risk of mode collapse |

Run this cell after loading both the VAE and GAN pre-trained models.

In [ ]:
from VAE import VAE

vae = VAE(latent_dim=2).to(device)
vae.load_model(path="AIBIP/06-Generative_Image_Models/models/vae_fashion_mnist.pth", device=device)
vae.eval()

n = 8
vae_samples = vae.sample(n, device=device).cpu()
gan_samples = ((model.generate(n, device=device) + 1) / 2).cpu()

fig, axes = plt.subplots(2, n, figsize=(14, 4))

for i in range(n):
    axes[0, i].imshow(vae_samples[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(gan_samples[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("VAE", fontsize=12)
axes[1, 0].set_ylabel("GAN", fontsize=12)
plt.suptitle("VAE vs. GAN — Fashion-MNIST samples", fontsize=12)
plt.tight_layout()
plt.show()

### Latent Space Interpolation

Even without a structured posterior, we can interpolate linearly between two noise vectors $z_a$ and $z_b$ in the generator's input space. Because the generator is a continuous function, nearby latent points decode to visually similar images.

This is less principled than VAE interpolation — there is no guarantee that the path between $z_a$ and $z_b$ stays in a region of high data density — but in practice it often produces smooth and meaningful transitions.

In [ ]:
model.eval()

torch.manual_seed(0)
z_a = torch.randn(1, LATENT_DIM, device=device)
z_b = torch.randn(1, LATENT_DIM, device=device)

n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = model.generator(z_interp).squeeze().cpu()
        img = (img + 1) / 2  # [-1, 1] -> [0, 1]
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle("Latent space interpolation — GAN Generator", y=1.05)
plt.tight_layout()
plt.show()

### Mode Collapse

A fundamental failure mode of GANs is **mode collapse**: the generator learns to produce only a small subset of the training distribution — often a single convincing image — because this is sufficient to fool the discriminator.

Signs in the loss curves:
- $\mathcal{L}_G$ drops sharply while $\mathcal{L}_D$ spikes — the generator found a fixed point that fools $D$.
- Generated samples show little diversity across different noise inputs.

We can detect diversity collapse by measuring the **pairwise pixel variance** across a batch of samples: low variance indicates the generator is producing near-identical images.

In [ ]:
model.eval()

n_check  = 64
z_check  = torch.randn(n_check, LATENT_DIM, device=device)

with torch.no_grad():
    samples_check = model.generator(z_check).cpu()  # (64, 1, 28, 28)

# Per-pixel variance across the batch — high variance = diverse samples
per_pixel_var = samples_check.var(dim=0).mean().item()
print(f"Mean per-pixel variance across {n_check} samples: {per_pixel_var:.4f}")
print("(Values near 0 indicate mode collapse; healthy GANs typically show > 0.05)")

# Visual diversity check: plot all 64 samples on a grid
grid = samples_check[:64]
grid = (grid + 1) / 2

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(grid[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle(f"64 GAN samples — per-pixel variance: {per_pixel_var:.4f}", fontsize=11)
plt.tight_layout()
plt.show()